In [ ]:
# ==============================================================
# R0 - Runtime / DuckDB lock-safe bootstrap
# ==============================================================
from pathlib import Path
import duckdb
import tomllib
import atexit

PROJECT_ROOT = Path.cwd()
CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"

if CONFIG_TOML_PATH.exists():
    with open(CONFIG_TOML_PATH, "rb") as f:
        SHARED_CONFIG = tomllib.load(f)
    print("Loaded shared config:", CONFIG_TOML_PATH)
else:
    SHARED_CONFIG = {
        "paths": {
            "data_dir": "content",
            "output_dir": "outputs_benchmark_survival",
            "tables_subdir": "tables",
            "metadata_subdir": "metadata",
            "data_output_subdir": "data",
            "duckdb_filename": "benchmark_survival.duckdb",
        },
        "benchmark": {
            "seed": 42,
            "test_size": 0.30,
            "early_window_weeks": 4,
            "main_enrollment_window_weeks": 4,
        },
    }
    print("Shared config TOML not found. Using defaults in-memory.")

paths_cfg = SHARED_CONFIG.get("paths", {})
SHARED_BENCHMARK_CONFIG = SHARED_CONFIG.get("benchmark", {})

def _resolve_project_path(raw_path: str) -> Path:
    p = Path(raw_path)
    return p if p.is_absolute() else PROJECT_ROOT / p

DATA_DIR = _resolve_project_path(paths_cfg.get("data_dir", "content"))
OUTPUT_DIR = _resolve_project_path(paths_cfg.get("output_dir", "outputs_benchmark_survival"))
TABLES_DIR = OUTPUT_DIR / paths_cfg.get("tables_subdir", "tables")
METADATA_DIR = OUTPUT_DIR / paths_cfg.get("metadata_subdir", "metadata")
DATA_OUTPUT_DIR = OUTPUT_DIR / paths_cfg.get("data_output_subdir", "data")
DUCKDB_PATH = OUTPUT_DIR / paths_cfg.get("duckdb_filename", "benchmark_survival.duckdb")

for p in [OUTPUT_DIR, TABLES_DIR, METADATA_DIR, DATA_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if "con" in globals():
    try:
        con.close()
        print("Closed previous DuckDB connection before reconnecting.")
    except Exception as _close_err:
        print(f"Warning while closing previous DuckDB connection: {_close_err}")

con = duckdb.connect(str(DUCKDB_PATH))

def _close_duckdb_connection() -> None:
    if "con" in globals():
        try:
            con.close()
            print("DuckDB connection closed.")
        except Exception:
            pass

if "_duckdb_close_registered" not in globals():
    atexit.register(_close_duckdb_connection)
    _duckdb_close_registered = True

print("Runtime context ready.")
print("- OUTPUT_DIR  :", OUTPUT_DIR)
print("- DUCKDB_PATH :", DUCKDB_PATH)

# C — Split design, audit, and modeling-ready partitions

## C1 — Event, Time, and Censoring Audit

In [ ]:
# ============================================================
# C1 — Event, Time, and Censoring Audit
# FINAL SUBSTITUTE VERSION
# ============================================================
# Purpose:
#   Audit the current survival operationalization before introducing
#   stronger split designs. This step does not alter the benchmark
#   data. It generates auditable evidence about:
#     - unit of analysis
#     - event definition
#     - censoring conventions
#     - zero-activity and pre-start activity cases
#     - temporal consistency
#     - course/presentation duration variation
#     - alignment between enrollment-level and person-period data
#     - concrete examples of flagged cases
#
# Expected inputs from previous cells:
#   - con
#   - enrollment_survival_ready table (from P5)
#   - courses table (if available)
#   - some person-period table (if available)
#
# Main outputs:
#   - table_p13_0_event_time_censoring_summary.csv
#   - table_p13_0_event_definition_breakdown.csv
#   - table_p13_0_temporal_consistency_checks.csv
#   - table_p13_0_zero_activity_breakdown.csv
#   - table_p13_0_course_duration_audit.csv
#   - table_p13_0_person_period_alignment.csv
#   - table_p13_0_duration_overrun_summary.csv
#   - table_p13_0_problem_examples.csv
#   - metadata_p13_0_event_time_censoring_audit.json
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C1 — Event, Time, and Censoring Audit")
print("=" * 70)
print("Substitute/final audit version.")
print("This step audits the current survival operationalization.")
print("It does not modify the event logic or the split.")
print("It generates auditable evidence for event definition,")
print("censoring conventions, temporal consistency, duration")
print("overruns, and enrollment/person-period alignment.")

# ------------------------------------------------------------
# 0) Runtime rehydration for standalone execution
# ------------------------------------------------------------
if "con" not in globals():
    PROJECT_ROOT = Path.cwd()
    CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"

    try:
        import tomllib
    except ModuleNotFoundError:
        tomllib = None

    if tomllib is not None and CONFIG_TOML_PATH.exists():
        with open(CONFIG_TOML_PATH, "rb") as f:
            SHARED_CONFIG = tomllib.load(f)
        print("Loaded shared config:", CONFIG_TOML_PATH)
    else:
        SHARED_CONFIG = globals().get("SHARED_CONFIG", {})
        if not SHARED_CONFIG:
            SHARED_CONFIG = {
                "paths": {
                    "output_dir": "outputs_benchmark_survival",
                    "tables_subdir": "tables",
                    "metadata_subdir": "metadata",
                    "duckdb_filename": "benchmark_survival.duckdb",
                }
            }
        print("Shared config TOML not found for C1. Using defaults in-memory.")

    paths_cfg = SHARED_CONFIG.get("paths", {})
    OUT_BASE = Path(paths_cfg.get("output_dir", "outputs_benchmark_survival"))
    if not OUT_BASE.is_absolute():
        OUT_BASE = PROJECT_ROOT / OUT_BASE
    OUT_TABLES = OUT_BASE / paths_cfg.get("tables_subdir", "tables")
    OUT_METADATA = OUT_BASE / paths_cfg.get("metadata_subdir", "metadata")
    DUCKDB_PATH = OUT_BASE / paths_cfg.get("duckdb_filename", "benchmark_survival.duckdb")

    OUT_TABLES.mkdir(parents=True, exist_ok=True)
    OUT_METADATA.mkdir(parents=True, exist_ok=True)

    import duckdb
    con = duckdb.connect(str(DUCKDB_PATH))
    print("Connected to DuckDB:", DUCKDB_PATH)
else:
    paths_cfg = globals().get("SHARED_CONFIG", {}).get("paths", {})
    OUT_BASE = globals().get("OUT_BASE", Path(paths_cfg.get("output_dir", "outputs_benchmark_survival")))
    OUT_TABLES = globals().get("OUT_TABLES", OUT_BASE / paths_cfg.get("tables_subdir", "tables"))
    OUT_METADATA = globals().get("OUT_METADATA", OUT_BASE / paths_cfg.get("metadata_subdir", "metadata"))
    OUT_TABLES.mkdir(parents=True, exist_ok=True)
    OUT_METADATA.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1) Resolve required enrollment-level survival table
# ------------------------------------------------------------
available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

if "enrollment_survival_ready" not in available_tables:
    raise RuntimeError(
        "DuckDB table 'enrollment_survival_ready' not found. "
        "Run P5 before P13.0."
    )

enr = con.execute("""
    SELECT *
    FROM enrollment_survival_ready
""").fetchdf()

required_enr_cols = [
    "id_student", "code_module", "code_presentation",
    "final_result", "date_registration", "date_unregistration",
    "is_withdrawn", "has_valid_unregistration_date",
    "event_observed", "is_withdrawn_without_valid_unregistration",
    "has_any_vle_activity", "max_vle_day", "n_vle_rows",
    "total_clicks_all_time", "t_event_week", "t_last_obs_week",
    "t_final_week", "used_zero_week_fallback_for_censoring"
]
missing_enr_cols = [c for c in required_enr_cols if c not in enr.columns]
if missing_enr_cols:
    raise KeyError(
        f"'enrollment_survival_ready' is missing required columns: {missing_enr_cols}"
    )

enr = enr.copy()
enr["enrollment_id"] = (
    enr["id_student"].astype(str)
    + "||" + enr["code_module"].astype(str)
    + "||" + enr["code_presentation"].astype(str)
)

# numeric coercions
date_reg = pd.to_numeric(enr["date_registration"], errors="coerce")
date_unreg = pd.to_numeric(enr["date_unregistration"], errors="coerce")
t_event = pd.to_numeric(enr["t_event_week"], errors="coerce")
t_last_obs = pd.to_numeric(enr["t_last_obs_week"], errors="coerce")
t_final = pd.to_numeric(enr["t_final_week"], errors="coerce")
max_vle_day = pd.to_numeric(enr["max_vle_day"], errors="coerce")
n_vle_rows = pd.to_numeric(enr["n_vle_rows"], errors="coerce")
total_clicks_all_time = pd.to_numeric(enr["total_clicks_all_time"], errors="coerce")

# ------------------------------------------------------------
# 2) Enrollment-level core summary
# ------------------------------------------------------------
n_total = len(enr)
n_unique_enrollment = enr["enrollment_id"].nunique()
n_unique_student = enr["id_student"].nunique()

n_withdrawn_total = int(pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0).sum())
n_event_observed = int(pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0).sum())
n_withdrawn_without_valid_unreg = int(
    pd.to_numeric(enr["is_withdrawn_without_valid_unregistration"], errors="coerce").fillna(0).sum()
)
n_nonwithdrawn = int((pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) == 0).sum())

n_any_vle = int(pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0).sum())
n_no_vle = int((pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 0).sum())
n_zero_week_fallback = int(
    pd.to_numeric(enr["used_zero_week_fallback_for_censoring"], errors="coerce").fillna(0).sum()
)

n_negative_max_vle_day = int((max_vle_day < 0).fillna(False).sum())
n_negative_t_last_obs = int((t_last_obs < 0).fillna(False).sum())
n_negative_t_final = int((t_final < 0).fillna(False).sum())

table_p13_0_event_time_censoring_summary = pd.DataFrame([{
    "n_total_enrollments": int(n_total),
    "n_unique_enrollments": int(n_unique_enrollment),
    "n_unique_students": int(n_unique_student),
    "enrollment_key_unique": bool(n_total == n_unique_enrollment),
    "n_withdrawn_total": n_withdrawn_total,
    "n_event_observed": n_event_observed,
    "event_rate": float(pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0).mean()),
    "n_withdrawn_without_valid_unregistration": n_withdrawn_without_valid_unreg,
    "share_withdrawn_without_valid_unregistration_among_withdrawn":
        float(n_withdrawn_without_valid_unreg / n_withdrawn_total) if n_withdrawn_total > 0 else np.nan,
    "n_nonwithdrawn": n_nonwithdrawn,
    "n_with_any_vle_activity": n_any_vle,
    "n_without_vle_activity": n_no_vle,
    "share_without_vle_activity": float(n_no_vle / n_total) if n_total > 0 else np.nan,
    "n_used_zero_week_fallback_for_censoring": n_zero_week_fallback,
    "share_zero_week_fallback": float(n_zero_week_fallback / n_total) if n_total > 0 else np.nan,
    "n_negative_max_vle_day": n_negative_max_vle_day,
    "n_negative_t_last_obs_week": n_negative_t_last_obs,
    "n_negative_t_final_week": n_negative_t_final,
    "max_t_event_week": t_event.max(),
    "max_t_last_obs_week": t_last_obs.max(),
    "max_t_final_week": t_final.max(),
}])

# ------------------------------------------------------------
# 3) Event-definition breakdown
# ------------------------------------------------------------
table_p13_0_event_definition_breakdown = (
    enr.groupby(
        [
            "final_result",
            "is_withdrawn",
            "has_valid_unregistration_date",
            "event_observed",
            "is_withdrawn_without_valid_unregistration",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n")
    .sort_values(
        [
            "final_result",
            "is_withdrawn",
            "has_valid_unregistration_date",
            "event_observed",
            "is_withdrawn_without_valid_unregistration",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4) Temporal consistency checks
# ------------------------------------------------------------
checks = []


### Funcao: add_check

Definicao isolada de add_check para reutilizacao nas etapas seguintes.


In [ ]:

def add_check(name: str, mask: pd.Series):
    checks.append({
        "check_name": name,
        "n_flagged": int(mask.fillna(False).sum()),
        "share_flagged": float(mask.fillna(False).mean()) if len(mask) else np.nan
    })


In [ ]:

mask_event_observed_but_not_withdrawn = (
    (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) != 1)
)
mask_event_observed_but_missing_t_event_week = (
    (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
    & (t_event.isna())
)
mask_non_event_but_has_t_event_week = (
    (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 0)
    & (t_event.notna())
)
mask_withdrawn_without_valid_unregistration_but_event_observed = (
    (pd.to_numeric(enr["is_withdrawn_without_valid_unregistration"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
)
mask_valid_unregistration_before_registration = (
    date_unreg.notna() & date_reg.notna() & (date_unreg < date_reg)
)
mask_event_week_negative = t_event.notna() & (t_event < 0)
mask_last_obs_week_negative = t_last_obs.notna() & (t_last_obs < 0)
mask_final_week_negative = t_final.notna() & (t_final < 0)
mask_event_after_final_week = t_event.notna() & t_final.notna() & (t_event > t_final)
mask_last_obs_after_final_week = t_last_obs.notna() & t_final.notna() & (t_last_obs > t_final)
mask_has_any_vle_activity_but_missing_max_vle_day = (
    (pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 1)
    & (max_vle_day.isna())
)
mask_no_vle_activity_but_positive_last_obs_week = (
    (pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 0)
    & t_last_obs.notna() & (t_last_obs > 0)
)
mask_zero_week_fallback_used_but_has_vle_activity = (
    (pd.to_numeric(enr["used_zero_week_fallback_for_censoring"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 1)
)
mask_zero_week_fallback_used_but_event_observed = (
    (pd.to_numeric(enr["used_zero_week_fallback_for_censoring"], errors="coerce").fillna(0) == 1)
    & (pd.to_numeric(enr["event_observed"], errors="coerce").fillna(0) == 1)
)
mask_negative_max_vle_day = max_vle_day.notna() & (max_vle_day < 0)
mask_fail_with_withdrawn_flag = (
    enr["final_result"].astype(str).str.strip().eq("Fail")
    & (pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) == 1)
)
mask_withdrawn_without_withdrawn_label = (
    enr["final_result"].astype(str).str.strip().eq("Withdrawn")
    & (pd.to_numeric(enr["is_withdrawn"], errors="coerce").fillna(0) != 1)
)

add_check("event_observed_but_not_withdrawn", mask_event_observed_but_not_withdrawn)
add_check("event_observed_but_missing_t_event_week", mask_event_observed_but_missing_t_event_week)
add_check("non_event_but_has_t_event_week", mask_non_event_but_has_t_event_week)
add_check(
    "withdrawn_without_valid_unregistration_but_event_observed",
    mask_withdrawn_without_valid_unregistration_but_event_observed
)
add_check("valid_unregistration_before_registration", mask_valid_unregistration_before_registration)
add_check("event_week_negative", mask_event_week_negative)
add_check("last_obs_week_negative", mask_last_obs_week_negative)
add_check("final_week_negative", mask_final_week_negative)
add_check("event_after_final_week", mask_event_after_final_week)
add_check("last_obs_after_final_week", mask_last_obs_after_final_week)
add_check("has_any_vle_activity_but_missing_max_vle_day", mask_has_any_vle_activity_but_missing_max_vle_day)
add_check("no_vle_activity_but_positive_last_obs_week", mask_no_vle_activity_but_positive_last_obs_week)
add_check("zero_week_fallback_used_but_has_vle_activity", mask_zero_week_fallback_used_but_has_vle_activity)
add_check("zero_week_fallback_used_but_event_observed", mask_zero_week_fallback_used_but_event_observed)
add_check("negative_max_vle_day", mask_negative_max_vle_day)
add_check("fail_with_withdrawn_flag", mask_fail_with_withdrawn_flag)
add_check("withdrawn_without_withdrawn_flag", mask_withdrawn_without_withdrawn_label)

table_p13_0_temporal_consistency_checks = pd.DataFrame(checks)

# ------------------------------------------------------------
# 5) Zero-activity / sparse-activity / pre-start activity breakdown
# ------------------------------------------------------------
enr["n_vle_rows_num"] = n_vle_rows.fillna(0)
enr["total_clicks_all_time_num"] = total_clicks_all_time.fillna(0)

enr["activity_profile"] = np.select(
    [
        pd.to_numeric(enr["has_any_vle_activity"], errors="coerce").fillna(0) == 0,
        max_vle_day.notna() & (max_vle_day < 0),
        enr["n_vle_rows_num"] <= 1,
        enr["n_vle_rows_num"].between(2, 3, inclusive="both"),
        enr["n_vle_rows_num"] >= 4,
    ],
    [
        "no_vle_activity",
        "pre_start_only_activity",
        "1_vle_row",
        "2_to_3_vle_rows",
        "4plus_vle_rows",
    ],
    default="other"
)

table_p13_0_zero_activity_breakdown = (
    enr.groupby(
        ["activity_profile", "event_observed", "used_zero_week_fallback_for_censoring"],
        dropna=False
    )
    .size()
    .reset_index(name="n")
    .sort_values(["activity_profile", "event_observed", "used_zero_week_fallback_for_censoring"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6) Course/presentation duration audit
# ------------------------------------------------------------
if "courses" in available_tables:
    courses_df = con.execute("""
        SELECT *
        FROM courses
    """).fetchdf()

    possible_duration_cols = [
        "module_presentation_length",
        "length",
        "course_length",
        "presentation_length",
    ]
    duration_col = next((c for c in possible_duration_cols if c in courses_df.columns), None)

    if duration_col is not None:
        course_duration_df = courses_df[["code_module", "code_presentation", duration_col]].copy()
        course_duration_df = course_duration_df.rename(columns={duration_col: "official_course_length_days"})
        course_duration_df["official_course_length_weeks_floor"] = np.floor(
            pd.to_numeric(course_duration_df["official_course_length_days"], errors="coerce") / 7.0
        )
    else:
        course_duration_df = courses_df[["code_module", "code_presentation"]].drop_duplicates().copy()
        course_duration_df["official_course_length_days"] = np.nan
        course_duration_df["official_course_length_weeks_floor"] = np.nan
else:
    course_duration_df = enr[["code_module", "code_presentation"]].drop_duplicates().copy()
    course_duration_df["official_course_length_days"] = np.nan
    course_duration_df["official_course_length_weeks_floor"] = np.nan

obs_duration_df = (
    enr.groupby(["code_module", "code_presentation"], dropna=False)
    .agg(
        n_enrollments=("enrollment_id", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_observed_t_final_week=("t_final_week", "max"),
        median_observed_t_final_week=("t_final_week", "median"),
        max_observed_t_event_week=("t_event_week", "max"),
        median_observed_t_last_obs_week=("t_last_obs_week", "median"),
    )
    .reset_index()
)

table_p13_0_course_duration_audit = (
    obs_duration_df.merge(
        course_duration_df,
        on=["code_module", "code_presentation"],
        how="left",
        validate="one_to_one"
    )
    .sort_values(["code_module", "code_presentation"])
    .reset_index(drop=True)
)

table_p13_0_duration_overrun_summary = table_p13_0_course_duration_audit.copy()
table_p13_0_duration_overrun_summary["observed_minus_official_weeks"] = (
    pd.to_numeric(table_p13_0_duration_overrun_summary["max_observed_t_final_week"], errors="coerce")
    - pd.to_numeric(table_p13_0_duration_overrun_summary["official_course_length_weeks_floor"], errors="coerce")
)
table_p13_0_duration_overrun_summary["has_duration_overrun"] = (
    table_p13_0_duration_overrun_summary["observed_minus_official_weeks"] > 0
)
table_p13_0_duration_overrun_summary = (
    table_p13_0_duration_overrun_summary
    .sort_values(
        ["has_duration_overrun", "observed_minus_official_weeks", "code_module", "code_presentation"],
        ascending=[False, False, True, True]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7) Enrollment vs person-period alignment audit
# ------------------------------------------------------------
# Search more robustly for a candidate person-period table
candidate_pp_tables = []
for name in sorted(available_tables):
    lname = name.lower()
    if ("person" in lname and "period" in lname) or ("pp_" in lname) or ("hazard_ready" in lname):
        candidate_pp_tables.append(name)


### Funcao: audit_person_period_table

Definicao isolada de audit_person_period_table para reutilizacao nas etapas seguintes.


In [ ]:

def audit_person_period_table(table_name: str, enr_df: pd.DataFrame):
    pp = con.execute(f"SELECT * FROM {table_name}").fetchdf()

    required_pp_cols = ["id_student", "code_module", "code_presentation", "week"]
    missing_pp_cols = [c for c in required_pp_cols if c not in pp.columns]

    result = {
        "person_period_table_found": True,
        "person_period_table_name": table_name,
        "required_columns_present": len(missing_pp_cols) == 0,
        "missing_columns": ", ".join(missing_pp_cols),
        "n_person_period_rows": int(len(pp)),
        "n_unique_person_period_enrollments": np.nan,
        "max_week_in_person_period": np.nan,
        "max_t_final_week_in_enrollment_table": pd.to_numeric(enr_df["t_final_week"], errors="coerce").max(),
        "all_enrollment_ids_covered_in_person_period": np.nan,
        "n_missing_enrollment_ids_in_person_period": np.nan,
        "share_enrollments_with_pp_max_week_equal_t_final_week": np.nan,
    }

    if len(missing_pp_cols) > 0:
        return pd.DataFrame([result])

    pp = pp.copy()
    pp["enrollment_id"] = (
        pp["id_student"].astype(str)
        + "||" + pp["code_module"].astype(str)
        + "||" + pp["code_presentation"].astype(str)
    )

    pp_cov = (
        pp.groupby("enrollment_id", dropna=False)
        .agg(
            pp_min_week=("week", "min"),
            pp_max_week=("week", "max"),
            pp_n_rows=("week", "size"),
        )
        .reset_index()
    )

    enr_cov = enr_df[["enrollment_id", "t_final_week", "event_observed"]].copy()
    enr_cov["t_final_week"] = pd.to_numeric(enr_cov["t_final_week"], errors="coerce")

    merged_cov = enr_cov.merge(pp_cov, on="enrollment_id", how="left", validate="one_to_one")
    missing_pp_enrollments = merged_cov["pp_n_rows"].isna()

    aligned_final_week = (
        merged_cov["pp_max_week"].notna()
        & merged_cov["t_final_week"].notna()
        & (merged_cov["pp_max_week"] == merged_cov["t_final_week"])
    )

    result.update({
        "n_unique_person_period_enrollments": int(pp["enrollment_id"].nunique()),
        "max_week_in_person_period": pd.to_numeric(pp["week"], errors="coerce").max(),
        "all_enrollment_ids_covered_in_person_period": bool((~missing_pp_enrollments).all()),
        "n_missing_enrollment_ids_in_person_period": int(missing_pp_enrollments.sum()),
        "share_enrollments_with_pp_max_week_equal_t_final_week":
            float(aligned_final_week.mean()) if len(aligned_final_week) else np.nan,
    })
    return pd.DataFrame([result])


In [ ]:

pp_alignment_frames = []
for tbl in candidate_pp_tables:
    try:
        pp_alignment_frames.append(audit_person_period_table(tbl, enr))
    except Exception as e:
        pp_alignment_frames.append(pd.DataFrame([{
            "person_period_table_found": True,
            "person_period_table_name": tbl,
            "required_columns_present": False,
            "missing_columns": f"ERROR: {str(e)}",
            "n_person_period_rows": np.nan,
            "n_unique_person_period_enrollments": np.nan,
            "max_week_in_person_period": np.nan,
            "max_t_final_week_in_enrollment_table": pd.to_numeric(enr["t_final_week"], errors="coerce").max(),
            "all_enrollment_ids_covered_in_person_period": np.nan,
            "n_missing_enrollment_ids_in_person_period": np.nan,
            "share_enrollments_with_pp_max_week_equal_t_final_week": np.nan,
        }]))

if len(pp_alignment_frames) > 0:
    table_p13_0_person_period_alignment = pd.concat(pp_alignment_frames, ignore_index=True)
else:
    table_p13_0_person_period_alignment = pd.DataFrame([{
        "person_period_table_found": False,
        "person_period_table_name": "",
        "required_columns_present": False,
        "missing_columns": "",
        "n_person_period_rows": np.nan,
        "n_unique_person_period_enrollments": np.nan,
        "max_week_in_person_period": np.nan,
        "max_t_final_week_in_enrollment_table": pd.to_numeric(enr["t_final_week"], errors="coerce").max(),
        "all_enrollment_ids_covered_in_person_period": np.nan,
        "n_missing_enrollment_ids_in_person_period": np.nan,
        "share_enrollments_with_pp_max_week_equal_t_final_week": np.nan,
    }])

# ------------------------------------------------------------
# 8) Problem examples
# ------------------------------------------------------------
problem_frames = []


### Funcao: collect_examples

Definicao isolada de collect_examples para reutilizacao nas etapas seguintes.


In [ ]:

def collect_examples(mask: pd.Series, label: str, max_rows: int = 20):
    cols = [
        "id_student", "code_module", "code_presentation", "final_result",
        "date_registration", "date_unregistration",
        "is_withdrawn", "has_valid_unregistration_date", "event_observed",
        "is_withdrawn_without_valid_unregistration",
        "has_any_vle_activity", "max_vle_day", "n_vle_rows",
        "total_clicks_all_time", "t_event_week", "t_last_obs_week", "t_final_week",
        "used_zero_week_fallback_for_censoring"
    ]
    use_cols = [c for c in cols if c in enr.columns]
    tmp = enr.loc[mask.fillna(False), use_cols].copy().head(max_rows)
    if len(tmp) > 0:
        tmp.insert(0, "problem_type", label)
        problem_frames.append(tmp)


In [ ]:

collect_examples(mask_last_obs_week_negative, "last_obs_week_negative")
collect_examples(mask_final_week_negative, "final_week_negative")
collect_examples(mask_last_obs_after_final_week, "last_obs_after_final_week")
collect_examples(mask_negative_max_vle_day, "negative_max_vle_day")
collect_examples(mask_fail_with_withdrawn_flag, "fail_with_withdrawn_flag")
collect_examples(mask_withdrawn_without_withdrawn_label, "withdrawn_without_withdrawn_flag")

duration_overrun_pairs = table_p13_0_duration_overrun_summary.loc[
    table_p13_0_duration_overrun_summary["has_duration_overrun"] == True,
    ["code_module", "code_presentation"]
].drop_duplicates()

if len(duration_overrun_pairs) > 0:
    overrun_keys = set(
        duration_overrun_pairs["code_module"].astype(str) + "||" + duration_overrun_pairs["code_presentation"].astype(str)
    )
    pair_mask = (
        enr["code_module"].astype(str) + "||" + enr["code_presentation"].astype(str)
    ).isin(overrun_keys)
    collect_examples(pair_mask, "duration_overrun_context_examples", max_rows=20)

if len(problem_frames) > 0:
    table_p13_0_problem_examples = pd.concat(problem_frames, ignore_index=True)
else:
    table_p13_0_problem_examples = pd.DataFrame(columns=[
        "problem_type",
        "id_student", "code_module", "code_presentation", "final_result",
        "date_registration", "date_unregistration",
        "is_withdrawn", "has_valid_unregistration_date", "event_observed",
        "is_withdrawn_without_valid_unregistration",
        "has_any_vle_activity", "max_vle_day", "n_vle_rows",
        "total_clicks_all_time", "t_event_week", "t_last_obs_week", "t_final_week",
        "used_zero_week_fallback_for_censoring"
    ])

# ------------------------------------------------------------
# 9) Save outputs
# ------------------------------------------------------------
path_summary = OUT_TABLES / "table_p13_0_event_time_censoring_summary.csv"
path_event_breakdown = OUT_TABLES / "table_p13_0_event_definition_breakdown.csv"
path_temporal_checks = OUT_TABLES / "table_p13_0_temporal_consistency_checks.csv"
path_zero_activity = OUT_TABLES / "table_p13_0_zero_activity_breakdown.csv"
path_course_duration = OUT_TABLES / "table_p13_0_course_duration_audit.csv"
path_pp_alignment = OUT_TABLES / "table_p13_0_person_period_alignment.csv"
path_duration_overrun = OUT_TABLES / "table_p13_0_duration_overrun_summary.csv"
path_problem_examples = OUT_TABLES / "table_p13_0_problem_examples.csv"
path_metadata = OUT_METADATA / "metadata_p13_0_event_time_censoring_audit.json"

table_p13_0_event_time_censoring_summary.to_csv(path_summary, index=False)
table_p13_0_event_definition_breakdown.to_csv(path_event_breakdown, index=False)
table_p13_0_temporal_consistency_checks.to_csv(path_temporal_checks, index=False)
table_p13_0_zero_activity_breakdown.to_csv(path_zero_activity, index=False)
table_p13_0_course_duration_audit.to_csv(path_course_duration, index=False)
table_p13_0_person_period_alignment.to_csv(path_pp_alignment, index=False)
table_p13_0_duration_overrun_summary.to_csv(path_duration_overrun, index=False)
table_p13_0_problem_examples.to_csv(path_problem_examples, index=False)

metadata_p13_0 = {
    "step": "P13.0",
    "title": "Event, Time, and Censoring Audit",
    "version": "final_substitute",
    "purpose": (
        "Audit the current survival operationalization before stronger split "
        "design changes. This step does not alter the benchmark."
    ),
    "event_definition_under_audit": "event_observed = 1 iff final_result='Withdrawn' and valid date_unregistration >= 0",
    "censoring_convention_under_audit": (
        "Non-event enrollments are censored at the last observed VLE week; "
        "if no VLE activity is observed and no event is observed, a week-0 "
        "fallback is used."
    ),
    "candidate_person_period_tables_detected": candidate_pp_tables,
    "output_tables": [
        path_summary.as_posix(),
        path_event_breakdown.as_posix(),
        path_temporal_checks.as_posix(),
        path_zero_activity.as_posix(),
        path_course_duration.as_posix(),
        path_pp_alignment.as_posix(),
        path_duration_overrun.as_posix(),
        path_problem_examples.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata_p13_0, f, indent=2)

# ------------------------------------------------------------
# 10) Display key outputs
# ------------------------------------------------------------
print("\nMain summary:")
display(table_p13_0_event_time_censoring_summary)

print("\nEvent definition breakdown:")
display(table_p13_0_event_definition_breakdown)

print("\nTemporal consistency checks:")
display(table_p13_0_temporal_consistency_checks)

print("\nZero / sparse / pre-start activity breakdown:")
display(table_p13_0_zero_activity_breakdown)

print("\nCourse/presentation duration audit:")
display(table_p13_0_course_duration_audit.head(20))

print("\nDuration overrun summary (top rows):")
display(table_p13_0_duration_overrun_summary.head(20))

print("\nPerson-period alignment audit:")
display(table_p13_0_person_period_alignment)

print("\nProblem examples:")
display(table_p13_0_problem_examples.head(50))

print("\nSaved:")
print("-", path_summary.resolve())
print("-", path_event_breakdown.resolve())
print("-", path_temporal_checks.resolve())
print("-", path_zero_activity.resolve())
print("-", path_course_duration.resolve())
print("-", path_pp_alignment.resolve())
print("-", path_duration_overrun.resolve())
print("-", path_problem_examples.resolve())
print("-", path_metadata.resolve())


## C2 — Apply Temporal Split and Leakage Control

In [ ]:
# ============================================================
# C2 — Apply Temporal Split and Leakage Control
# ============================================================
# Methodological note:
# This step defines the main benchmark split at the enrollment level,
# stratified by event status and coarse event-time bucket.
# The leakage audit in this step is restricted to enrollment-level
# identity leakage and propagation consistency across derived benchmark
# tables. Module/presentation overlap is not treated here as leakage by
# definition; it will be audited separately in a dedicated contextual audit.

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

print("=" * 70)
print("C2 — Apply Temporal Split and Leakage Control")
print("=" * 70)
print("Methodological note: this step defines the main benchmark split at the")
print("enrollment level, stratified by event status and coarse event-time bucket.")
print("The leakage audit in this step is restricted to enrollment-level identity")
print("leakage and propagation consistency across derived benchmark tables.")
print("Module/presentation overlap is not treated here as leakage by definition;")
print("it will be audited separately in a dedicated contextual split audit.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
TEST_SIZE = 0.30
RANDOM_STATE = 42
N_DURATION_BUCKETS = 4
MIN_ROWS_FOR_CANONICAL_BACKBONE = 1000

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)


### Funcao: build_enrollment_id

Definicao isolada de build_enrollment_id para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def build_enrollment_id(df: pd.DataFrame) -> pd.Series:
    required_cols = ["id_student", "code_module", "code_presentation"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns required to build enrollment_id: {missing}")
    return (
        df["id_student"].astype(str)
        + "||" + df["code_module"].astype(str)
        + "||" + df["code_presentation"].astype(str)
    )


### Funcao: make_duration_buckets

Definicao isolada de make_duration_buckets para reutilizacao nas etapas seguintes.


In [ ]:


def make_duration_buckets(duration_series: pd.Series, n_buckets: int = 4) -> pd.Series:
    s = pd.to_numeric(duration_series, errors="coerce").fillna(-1)

    try:
        buckets = pd.qcut(s, q=n_buckets, duplicates="drop")
        return buckets.astype(str)
    except Exception:
        ranks = s.rank(method="average", pct=True)
        edges = np.linspace(0, 1, n_buckets + 1)
        labels = [f"bucket_{i+1}" for i in range(n_buckets)]
        return pd.cut(
            ranks,
            bins=edges,
            labels=labels,
            include_lowest=True
        ).astype(str)


### Funcao: attach_split_assignment

Definicao isolada de attach_split_assignment para reutilizacao nas etapas seguintes.


In [ ]:


def attach_split_assignment(df: pd.DataFrame, split_assignment: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "enrollment_id" not in out.columns:
        out["enrollment_id"] = build_enrollment_id(out)
    return out.merge(
        split_assignment[["enrollment_id", "split"]],
        on="enrollment_id",
        how="left",
        validate="many_to_one",
    )


### Funcao: summarize_split

Definicao isolada de summarize_split para reutilizacao nas etapas seguintes.


In [ ]:


def summarize_split(df_enroll: pd.DataFrame) -> pd.DataFrame:
    return (
        df_enroll.groupby("split", dropna=False)
        .agg(
            n_enrollments=("enrollment_id", "nunique"),
            n_events=("event", "sum"),
            event_rate=("event", "mean"),
            min_duration=("duration", "min"),
            median_duration=("duration", "median"),
            max_duration=("duration", "max"),
        )
        .reset_index()
    )


### Funcao: summarize_split_by_bucket

Definicao isolada de summarize_split_by_bucket para reutilizacao nas etapas seguintes.


In [ ]:


def summarize_split_by_bucket(df_enroll: pd.DataFrame) -> pd.DataFrame:
    return (
        df_enroll.groupby(["split", "event", "duration_bucket"], dropna=False)
        .agg(n_enrollments=("enrollment_id", "nunique"))
        .reset_index()
        .sort_values(["split", "event", "duration_bucket"])
        .reset_index(drop=True)
    )


### Funcao: name_penalty

Larger penalty = less likely to be canonical benchmark backbone.


In [ ]:


def name_penalty(name: str) -> int:
    """
    Larger penalty = less likely to be canonical benchmark backbone.
    """
    n = name.lower()

    penalty = 0

    # Strong negative cues
    bad_patterns = [
        r"^sample",
        r"sample_",
        r"^cols_",
        r"^perm_",
        r"perm_",
        r"debug",
        r"toy",
        r"mock",
        r"tmp",
        r"temp",
        r"copy",
        r"preview",
        r"head",
        r"subset",
    ]
    for pat in bad_patterns:
        if re.search(pat, n):
            penalty += 100

    # Test/train specific views are less canonical than a full backbone
    if n.startswith("train_") or n.endswith("_train") or "_train_" in n:
        penalty += 20
    if n.startswith("test_") or n.endswith("_test") or "_test_" in n:
        penalty += 20

    # Positive cues
    good_patterns = [
        r"backbone",
        r"enrollment",
        r"benchmark",
        r"master",
        r"base_df",
        r"analysis_df",
    ]
    for pat in good_patterns:
        if re.search(pat, n):
            penalty -= 10

    return penalty


### Funcao: find_candidate_enrollment_backbones

Definicao isolada de find_candidate_enrollment_backbones para reutilizacao nas etapas seguintes.


In [ ]:


def find_candidate_enrollment_backbones(globals_dict: dict) -> pd.DataFrame:
    required_cols = {
        "id_student", "code_module", "code_presentation", "event", "duration"
    }

    candidates = []

    for name, obj in globals_dict.items():
        if not isinstance(obj, pd.DataFrame):
            continue

        cols = set(obj.columns)
        if not required_cols.issubset(cols):
            continue

        try:
            tmp = obj.copy()
            tmp["enrollment_id"] = build_enrollment_id(tmp)

            n_rows = len(tmp)
            n_unique_enrollments = tmp["enrollment_id"].nunique()
            duplicate_rows = n_rows - n_unique_enrollments
            duplicate_rate = duplicate_rows / n_rows if n_rows > 0 else np.nan

            event_numeric_share = pd.to_numeric(tmp["event"], errors="coerce").notna().mean()
            duration_numeric_share = pd.to_numeric(tmp["duration"], errors="coerce").notna().mean()

            candidates.append({
                "name": name,
                "n_rows": n_rows,
                "n_unique_enrollments": n_unique_enrollments,
                "duplicate_rows": duplicate_rows,
                "duplicate_rate": duplicate_rate,
                "event_numeric_share": event_numeric_share,
                "duration_numeric_share": duration_numeric_share,
                "name_penalty": name_penalty(name),
                "large_enough": n_rows >= MIN_ROWS_FOR_CANONICAL_BACKBONE,
            })
        except Exception:
            continue

    if len(candidates) == 0:
        return pd.DataFrame()

    cand_df = pd.DataFrame(candidates)

    # Prefer plausible real benchmark tables first
    plausible_df = cand_df[cand_df["large_enough"]].copy()

    if not plausible_df.empty:
        cand_df = plausible_df

    cand_df = cand_df.sort_values(
        by=[
            "event_numeric_share",
            "duration_numeric_share",
            "duplicate_rate",
            "name_penalty",
            "n_rows",
            "n_unique_enrollments",
        ],
        ascending=[False, False, True, True, False, False]
    ).reset_index(drop=True)

    return cand_df


In [ ]:


# ------------------------------------------------------------
# Resolve main enrollment-level dataframe
# ------------------------------------------------------------
candidate_summary = find_candidate_enrollment_backbones(globals())

if candidate_summary.empty:
    raise ValueError(
        "Could not find any plausible enrollment-level benchmark DataFrame in globals() "
        "with columns {id_student, code_module, code_presentation, event, duration}."
    )

print("\nCandidate enrollment backbones found:")
print(candidate_summary.head(20).to_string(index=False))

resolved_name = candidate_summary.iloc[0]["name"]
enroll_df = globals()[resolved_name].copy()

print(f"\nResolved canonical enrollment backbone from: {resolved_name}")
print(f"Rows in resolved enrollment backbone before deduplication check: {len(enroll_df):,}")

# ------------------------------------------------------------
# Enforce one row per enrollment
# ------------------------------------------------------------
if "enrollment_id" not in enroll_df.columns:
    enroll_df["enrollment_id"] = build_enrollment_id(enroll_df)

dup_counts = enroll_df["enrollment_id"].value_counts()
n_duplicate_ids = int((dup_counts > 1).sum())

if n_duplicate_ids > 0:
    print(f"Detected {n_duplicate_ids:,} duplicated enrollment_ids; keeping first occurrence.")
    enroll_df = (
        enroll_df.sort_values(["enrollment_id"])
        .drop_duplicates(subset=["enrollment_id"], keep="first")
        .reset_index(drop=True)
    )

print(f"Unique enrollments used for splitting: {enroll_df['enrollment_id'].nunique():,}")

# ------------------------------------------------------------
# Basic cleaning
# ------------------------------------------------------------
enroll_df["event"] = pd.to_numeric(enroll_df["event"], errors="coerce").fillna(0).astype(int)
enroll_df["duration"] = pd.to_numeric(enroll_df["duration"], errors="coerce")

before_clean = len(enroll_df)
enroll_df = enroll_df.dropna(subset=["duration"]).copy()
enroll_df["duration"] = enroll_df["duration"].astype(int)
after_clean = len(enroll_df)

if before_clean != after_clean:
    print(f"Dropped {before_clean - after_clean:,} enrollments due to missing duration.")

if len(enroll_df) < 10:
    raise ValueError(
        f"Resolved enrollment backbone '{resolved_name}' has only {len(enroll_df)} rows "
        "after cleaning, which is not plausible for the benchmark split."
    )

# ------------------------------------------------------------
# Create stratification label
# ------------------------------------------------------------
enroll_df["duration_bucket"] = make_duration_buckets(
    enroll_df["duration"],
    n_buckets=N_DURATION_BUCKETS
)
enroll_df["strata_label"] = (
    "event_" + enroll_df["event"].astype(str)
    + "__"
    + enroll_df["duration_bucket"].astype(str)
)

print("\nStratification label distribution (top 10):")
print(enroll_df["strata_label"].value_counts().head(10))

# ------------------------------------------------------------
# Split at enrollment level
# ------------------------------------------------------------
sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

X_dummy = np.zeros(len(enroll_df))
y_strata = enroll_df["strata_label"].astype(str).to_numpy()

train_idx, test_idx = next(sss.split(X_dummy, y_strata))

enroll_df["split"] = "train"
enroll_df.loc[enroll_df.index[test_idx], "split"] = "test"

split_assignment = enroll_df[
    ["enrollment_id", "id_student", "code_module", "code_presentation",
     "event", "duration", "duration_bucket", "strata_label", "split"]
].copy()

# ------------------------------------------------------------
# Main split summaries
# ------------------------------------------------------------
table_split_summary = summarize_split(enroll_df)
table_split_bucket_summary = summarize_split_by_bucket(enroll_df)

print("\nEnrollment split summary:")
print(table_split_summary.to_string(index=False))

print("\nEnrollment split bucket summary (head):")
print(table_split_bucket_summary.head(20).to_string(index=False))

# ------------------------------------------------------------
# Identity leakage audit (enrollment-level only)
# ------------------------------------------------------------
train_ids = set(split_assignment.loc[split_assignment["split"] == "train", "enrollment_id"])
test_ids = set(split_assignment.loc[split_assignment["split"] == "test", "enrollment_id"])
overlap_ids = train_ids.intersection(test_ids)

table_leakage_check = pd.DataFrame(
    [{
        "n_train_enrollments": len(train_ids),
        "n_test_enrollments": len(test_ids),
        "n_enrollments_checked": len(train_ids.union(test_ids)),
        "n_enrollments_with_leakage": len(overlap_ids),
        "identity_leakage_detected": len(overlap_ids) > 0,
    }]
)

print("\nEnrollment identity leakage check:")
print(table_leakage_check.to_string(index=False))

print("\nInterpretation:")
print("- The main benchmark split is enrollment-level.")
print("- Zero enrollment identity leakage was detected across train/test.")
print("- This check does not yet evaluate contextual overlap in code_module,")
print("  code_presentation, or module-presentation combinations.")
print("- Those contextual overlap diagnostics belong to the next dedicated step.")

# ------------------------------------------------------------
# Propagation check to derived dataframes (if available)
# ------------------------------------------------------------
derived_df_candidates = [
    "pp_df",
    "person_period_df",
    "df_person_period",
    "linear_df",
    "neural_df",
    "cox_df",
    "deepsurv_df",
]

propagation_rows = []

for name in derived_df_candidates:
    if name not in globals():
        continue
    df_obj = globals()[name]
    if not isinstance(df_obj, pd.DataFrame):
        continue

    try:
        df_tmp = df_obj.copy()

        if "enrollment_id" not in df_tmp.columns:
            required = {"id_student", "code_module", "code_presentation"}
            if required.issubset(df_tmp.columns):
                df_tmp["enrollment_id"] = build_enrollment_id(df_tmp)
            else:
                propagation_rows.append({
                    "table_name": name,
                    "n_rows": len(df_tmp),
                    "n_unique_enrollments": np.nan,
                    "n_missing_split_labels": np.nan,
                    "propagation_ok": False,
                    "note": "Could not derive enrollment_id"
                })
                continue

        df_attached = attach_split_assignment(df_tmp, split_assignment)
        n_missing = int(df_attached["split"].isna().sum())

        propagation_rows.append({
            "table_name": name,
            "n_rows": len(df_attached),
            "n_unique_enrollments": df_attached["enrollment_id"].nunique(),
            "n_missing_split_labels": n_missing,
            "propagation_ok": n_missing == 0,
            "note": "ok" if n_missing == 0 else "Some rows missing split label"
        })

    except Exception as e:
        propagation_rows.append({
            "table_name": name,
            "n_rows": len(df_obj),
            "n_unique_enrollments": np.nan,
            "n_missing_split_labels": np.nan,
            "propagation_ok": False,
            "note": f"Error: {type(e).__name__}: {e}"
        })

table_propagation_check = pd.DataFrame(propagation_rows)

if not table_propagation_check.empty:
    print("\nPropagation check across derived tables:")
    print(table_propagation_check.to_string(index=False))
else:
    print("\nPropagation check: no eligible derived tables found in globals().")

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------
path_split_assignment = OUT_TABLES / "table_enrollment_split_assignment.csv"
path_split_summary = OUT_TABLES / "table_enrollment_split_summary.csv"
path_split_bucket_summary = OUT_TABLES / "table_enrollment_split_bucket_summary.csv"
path_leakage_check = OUT_TABLES / "table_enrollment_split_leakage_check.csv"
path_propagation_check = OUT_TABLES / "table_enrollment_split_propagation_check.csv"
path_candidate_backbones = OUT_TABLES / "table_enrollment_backbone_candidates.csv"
path_protocol_note = OUT_METADATA / "enrollment_split_protocol_note.json"

split_assignment.to_csv(path_split_assignment, index=False)
table_split_summary.to_csv(path_split_summary, index=False)
table_split_bucket_summary.to_csv(path_split_bucket_summary, index=False)
table_leakage_check.to_csv(path_leakage_check, index=False)
candidate_summary.to_csv(path_candidate_backbones, index=False)

if not table_propagation_check.empty:
    table_propagation_check.to_csv(path_propagation_check, index=False)

split_protocol_note = {
    "step_id": "P13",
    "step_title": "Apply Temporal Split and Leakage Control (Revised v5)",
    "split_unit": "enrollment",
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "stratification": "event_status + coarse_event_time_bucket",
    "n_duration_buckets": N_DURATION_BUCKETS,
    "identity_leakage_check": "performed",
    "identity_leakage_result": (
        "no enrollment leakage detected"
        if len(overlap_ids) == 0
        else f"{len(overlap_ids)} overlapping enrollment_ids detected"
    ),
    "contextual_module_presentation_audit": "not performed in this step",
    "intended_followup_step": "C2.1 — Module/Presentation Leakage Audit",
    "resolved_enrollment_backbone": resolved_name,
    "n_enrollments_used": int(enroll_df["enrollment_id"].nunique()),
    "n_train_enrollments": int((enroll_df["split"] == "train").sum()),
    "n_test_enrollments": int((enroll_df["split"] == "test").sum()),
    "min_rows_for_canonical_backbone": MIN_ROWS_FOR_CANONICAL_BACKBONE,
}

with open(path_protocol_note, "w", encoding="utf-8") as f:
    json.dump(split_protocol_note, f, indent=2)

print("\nSaved:")
print(f"- {path_split_assignment}")
print(f"- {path_split_summary}")
print(f"- {path_split_bucket_summary}")
print(f"- {path_leakage_check}")
print(f"- {path_candidate_backbones}")
if not table_propagation_check.empty:
    print(f"- {path_propagation_check}")
print(f"- {path_protocol_note}")


### C2.1 — Module/Presentation Leakage Audit

In [ ]:
# ============================================================
# C2.1 — Module/Presentation Leakage Audit
# ============================================================
# Methodological note:
# This step does not redefine the benchmark split and does not classify
# module/presentation overlap as identity leakage. Its purpose is purely
# diagnostic: to characterize how much contextual overlap exists across
# train and test with respect to code_module, code_presentation, and the
# combined module-presentation context.

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.1 — Module/Presentation Leakage Audit")
print("=" * 70)
print("Methodological note: this step is diagnostic only.")
print("It does not change the benchmark split and does not classify")
print("module/presentation overlap as identity leakage.")
print("Its purpose is to quantify contextual overlap between train and test")
print("for code_module, code_presentation, and module-presentation pairs.")

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Resolve split assignment generated in P13
# ------------------------------------------------------------
if "split_assignment" not in globals():
    raise NameError(
        "split_assignment not found in globals(). Run P13 before P13.1."
    )

split_df = split_assignment.copy()

required_cols = [
    "enrollment_id", "id_student", "code_module", "code_presentation", "split"
]
missing_cols = [c for c in required_cols if c not in split_df.columns]
if missing_cols:
    raise KeyError(f"split_assignment is missing required columns: {missing_cols}")

split_df["code_module"] = split_df["code_module"].astype(str)
split_df["code_presentation"] = split_df["code_presentation"].astype(str)
split_df["module_presentation"] = (
    split_df["code_module"] + "||" + split_df["code_presentation"]
)

train_df = split_df.loc[split_df["split"] == "train"].copy()
test_df = split_df.loc[split_df["split"] == "test"].copy()

print(f"\nTrain enrollments: {len(train_df):,}")
print(f"Test enrollments:  {len(test_df):,}")

# ------------------------------------------------------------
# Set-based contextual overlap
# ------------------------------------------------------------
train_modules = set(train_df["code_module"].unique())
test_modules = set(test_df["code_module"].unique())

train_presentations = set(train_df["code_presentation"].unique())
test_presentations = set(test_df["code_presentation"].unique())

train_modpres = set(train_df["module_presentation"].unique())
test_modpres = set(test_df["module_presentation"].unique())

shared_modules = train_modules.intersection(test_modules)
shared_presentations = train_presentations.intersection(test_presentations)
shared_modpres = train_modpres.intersection(test_modpres)

table_context_overlap_summary = pd.DataFrame([
    {
        "context_level": "code_module",
        "n_train_unique": len(train_modules),
        "n_test_unique": len(test_modules),
        "n_shared": len(shared_modules),
        "train_share_seen_in_test": len(shared_modules) / len(train_modules) if len(train_modules) else np.nan,
        "test_share_seen_in_train": len(shared_modules) / len(test_modules) if len(test_modules) else np.nan,
    },
    {
        "context_level": "code_presentation",
        "n_train_unique": len(train_presentations),
        "n_test_unique": len(test_presentations),
        "n_shared": len(shared_presentations),
        "train_share_seen_in_test": len(shared_presentations) / len(train_presentations) if len(train_presentations) else np.nan,
        "test_share_seen_in_train": len(shared_presentations) / len(test_presentations) if len(test_presentations) else np.nan,
    },
    {
        "context_level": "module_presentation",
        "n_train_unique": len(train_modpres),
        "n_test_unique": len(test_modpres),
        "n_shared": len(shared_modpres),
        "train_share_seen_in_test": len(shared_modpres) / len(train_modpres) if len(train_modpres) else np.nan,
        "test_share_seen_in_train": len(shared_modpres) / len(test_modpres) if len(test_modpres) else np.nan,
    },
])

print("\nContextual overlap summary:")
print(table_context_overlap_summary.to_string(index=False))

# ------------------------------------------------------------
# Enrollment-level exposure to seen/unseen contexts
# ------------------------------------------------------------
test_df["module_seen_in_train"] = test_df["code_module"].isin(train_modules)
test_df["presentation_seen_in_train"] = test_df["code_presentation"].isin(train_presentations)
test_df["module_presentation_seen_in_train"] = test_df["module_presentation"].isin(train_modpres)

train_df["module_seen_in_test"] = train_df["code_module"].isin(test_modules)
train_df["presentation_seen_in_test"] = train_df["code_presentation"].isin(test_presentations)
train_df["module_presentation_seen_in_test"] = train_df["module_presentation"].isin(test_modpres)

table_enrollment_context_exposure = pd.DataFrame([
    {
        "split": "test",
        "n_enrollments": len(test_df),
        "share_module_seen_in_other_split": test_df["module_seen_in_train"].mean(),
        "share_presentation_seen_in_other_split": test_df["presentation_seen_in_train"].mean(),
        "share_module_presentation_seen_in_other_split": test_df["module_presentation_seen_in_train"].mean(),
    },
    {
        "split": "train",
        "n_enrollments": len(train_df),
        "share_module_seen_in_other_split": train_df["module_seen_in_test"].mean(),
        "share_presentation_seen_in_other_split": train_df["presentation_seen_in_test"].mean(),
        "share_module_presentation_seen_in_other_split": train_df["module_presentation_seen_in_test"].mean(),
    },
])

print("\nEnrollment-level contextual exposure:")
print(table_enrollment_context_exposure.to_string(index=False))

# ------------------------------------------------------------
# Breakdown by module-presentation pair
# ------------------------------------------------------------
table_module_presentation_split_counts = (
    split_df.groupby(["code_module", "code_presentation", "split"])
    .agg(n_enrollments=("enrollment_id", "nunique"))
    .reset_index()
    .pivot(index=["code_module", "code_presentation"], columns="split", values="n_enrollments")
    .fillna(0)
    .reset_index()
)

for col in ["train", "test"]:
    if col not in table_module_presentation_split_counts.columns:
        table_module_presentation_split_counts[col] = 0

table_module_presentation_split_counts["appears_in_both_splits"] = (
    (table_module_presentation_split_counts["train"] > 0)
    & (table_module_presentation_split_counts["test"] > 0)
)

table_module_presentation_split_counts = (
    table_module_presentation_split_counts
    .sort_values(
        ["appears_in_both_splits", "train", "test", "code_module", "code_presentation"],
        ascending=[False, False, False, True, True]
    )
    .reset_index(drop=True)
)

print("\nModule-presentation split counts (head):")
print(table_module_presentation_split_counts.head(20).to_string(index=False))

# ------------------------------------------------------------
# Interpretation block
# ------------------------------------------------------------
n_modpres_both = int(table_module_presentation_split_counts["appears_in_both_splits"].sum())
n_modpres_total = int(len(table_module_presentation_split_counts))

print("\nInterpretation:")
print("- This is a contextual overlap audit, not an identity leakage audit.")
print("- Enrollment identity leakage remains defined only at the enrollment_id level.")
print(f"- Module-presentation contexts appearing in both splits: {n_modpres_both:,} / {n_modpres_total:,}.")
print("- If overlap is high, the benchmark should be described as enrollment-level")
print("  with shared curricular context across splits, not as fully context-disjoint.")
print("- If overlap is low, this strengthens the claim that test evaluation includes")
print("  more contextual novelty at the module-presentation level.")

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------
path_context_overlap_summary = OUT_TABLES / "table_module_presentation_overlap_summary.csv"
path_context_exposure = OUT_TABLES / "table_module_presentation_context_exposure.csv"
path_modpres_counts = OUT_TABLES / "table_module_presentation_split_counts.csv"
path_context_metadata = OUT_METADATA / "module_presentation_overlap_audit_summary.json"

table_context_overlap_summary.to_csv(path_context_overlap_summary, index=False)
table_enrollment_context_exposure.to_csv(path_context_exposure, index=False)
table_module_presentation_split_counts.to_csv(path_modpres_counts, index=False)

context_metadata = {
    "step_id": "P13.1",
    "step_title": "Module/Presentation Leakage Audit",
    "diagnostic_only": True,
    "redefines_main_split": False,
    "identity_leakage_definition_changed": False,
    "n_train_enrollments": int(len(train_df)),
    "n_test_enrollments": int(len(test_df)),
    "n_unique_train_modules": int(len(train_modules)),
    "n_unique_test_modules": int(len(test_modules)),
    "n_shared_modules": int(len(shared_modules)),
    "n_unique_train_presentations": int(len(train_presentations)),
    "n_unique_test_presentations": int(len(test_presentations)),
    "n_shared_presentations": int(len(shared_presentations)),
    "n_unique_train_module_presentations": int(len(train_modpres)),
    "n_unique_test_module_presentations": int(len(test_modpres)),
    "n_shared_module_presentations": int(len(shared_modpres)),
    "n_module_presentations_in_both_splits": n_modpres_both,
    "n_total_module_presentations": n_modpres_total,
}
with open(path_context_metadata, "w", encoding="utf-8") as f:
    json.dump(context_metadata, f, indent=2)

print("\nSaved:")
print(f"- {path_context_overlap_summary}")
print(f"- {path_context_exposure}")
print(f"- {path_modpres_counts}")
print(f"- {path_context_metadata}")

### C2.2 — Consolidate Split and Context Audit for Paper Integration

In [ ]:
# ============================================================
# C2.2 — Consolidate Split and Context Audit for Paper Integration
# ============================================================
# Methodological note:
# This step does not alter the benchmark split. It consolidates the
# results of P13 and P13.1 into compact, paper-oriented artifacts that
# summarize (i) the enrollment-level split design, (ii) the identity
# leakage result, and (iii) the degree of shared curricular context
# across train and test.

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.2 — Consolidate Split and Context Audit for Paper Integration")
print("=" * 70)
print("Methodological note: this step does not alter the benchmark split.")
print("It consolidates the outputs of P13 and P13.1 into compact")
print("paper-oriented artifacts for manuscript integration.")

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_TABLES_PAPER_APPENDIX = OUT_TABLES / "paper_appendix"
OUT_METADATA = OUT_BASE / "metadata"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_TABLES_PAPER_APPENDIX.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Resolve required inputs from prior steps
# ------------------------------------------------------------
required_globals = ["split_assignment"]
missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise NameError(
        f"Missing required objects from prior steps: {missing_globals}. "
        "Run P13 and P13.1 before P13.2."
    )

split_df = split_assignment.copy()

required_split_cols = [
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "event", "duration", "duration_bucket", "split"
]
missing_split_cols = [c for c in required_split_cols if c not in split_df.columns]
if missing_split_cols:
    raise KeyError(f"split_assignment is missing required columns: {missing_split_cols}")

split_df["code_module"] = split_df["code_module"].astype(str)
split_df["code_presentation"] = split_df["code_presentation"].astype(str)
split_df["module_presentation"] = (
    split_df["code_module"] + "||" + split_df["code_presentation"]
)

train_df = split_df.loc[split_df["split"] == "train"].copy()
test_df = split_df.loc[split_df["split"] == "test"].copy()

# ------------------------------------------------------------
# Identity leakage summary
# ------------------------------------------------------------
train_ids = set(train_df["enrollment_id"])
test_ids = set(test_df["enrollment_id"])
overlap_ids = train_ids.intersection(test_ids)

identity_leakage_detected = len(overlap_ids) > 0

# ------------------------------------------------------------
# Contextual overlap summary
# ------------------------------------------------------------
train_modules = set(train_df["code_module"].unique())
test_modules = set(test_df["code_module"].unique())

train_presentations = set(train_df["code_presentation"].unique())
test_presentations = set(test_df["code_presentation"].unique())

train_modpres = set(train_df["module_presentation"].unique())
test_modpres = set(test_df["module_presentation"].unique())

shared_modules = train_modules.intersection(test_modules)
shared_presentations = train_presentations.intersection(test_presentations)
shared_modpres = train_modpres.intersection(test_modpres)

# ------------------------------------------------------------
# Paper-oriented compact summary table
# ------------------------------------------------------------
table_paper_appendix_split_context_audit = pd.DataFrame([{
    "split_unit": "enrollment",
    "stratification": "event_status + coarse_event_time_bucket",
    "n_total_enrollments": int(split_df["enrollment_id"].nunique()),
    "n_train_enrollments": int(train_df["enrollment_id"].nunique()),
    "n_test_enrollments": int(test_df["enrollment_id"].nunique()),
    "train_event_rate": float(pd.to_numeric(train_df["event"], errors="coerce").mean()),
    "test_event_rate": float(pd.to_numeric(test_df["event"], errors="coerce").mean()),
    "identity_leakage_detected": "no" if not identity_leakage_detected else "yes",
    "shared_modules": f"{len(shared_modules)}/{len(train_modules)}",
    "shared_presentations": f"{len(shared_presentations)}/{len(train_presentations)}",
    "shared_module_presentations": f"{len(shared_modpres)}/{len(train_modpres)}",
    "contextual_interpretation": (
        "shared curricular context across train and test"
        if len(shared_modpres) > 0
        else "context-disjoint curricular split"
    )
}])

print("\nPaper-oriented compact split/context audit:")
print(table_paper_appendix_split_context_audit.to_string(index=False))

# ------------------------------------------------------------
# Optional long-form commentary object for later freeze use
# ------------------------------------------------------------
table_paper_appendix_split_context_commentary = pd.DataFrame([{
    "object_id": "Appendix Table A5",
    "object_type": "table",
    "suggested_name": "Split and contextual overlap audit",
    "objective_comment_english": (
        "Document the benchmark split unit, stratification rule, identity leakage result, "
        "and degree of curricular-context overlap across train and test."
    ),
    "what_it_shows_comment_english": (
        "This table shows that the benchmark uses an enrollment-level split with no "
        "enrollment identity leakage, but with fully shared curricular context across "
        "train and test at the module, presentation, and module-presentation levels."
    )
}])

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------
path_compact = OUT_TABLES / "table_appendix_split_context_audit_compact.csv"
path_compact_paper = OUT_TABLES_PAPER_APPENDIX / "table_paper_appendix_split_context_audit.csv"
path_commentary = OUT_TABLES / "table_paper_appendix_split_context_commentary.csv"
path_metadata = OUT_METADATA / "split_context_audit_summary.json"

table_paper_appendix_split_context_audit.to_csv(path_compact, index=False)
table_paper_appendix_split_context_audit.to_csv(path_compact_paper, index=False)
table_paper_appendix_split_context_commentary.to_csv(path_commentary, index=False)

metadata = {
    "step_id": "P13.2",
    "step_title": "Consolidate Split and Context Audit for Paper Integration",
    "split_unit": "enrollment",
    "stratification": "event_status + coarse_event_time_bucket",
    "n_total_enrollments": int(split_df["enrollment_id"].nunique()),
    "n_train_enrollments": int(train_df["enrollment_id"].nunique()),
    "n_test_enrollments": int(test_df["enrollment_id"].nunique()),
    "identity_leakage_detected": bool(identity_leakage_detected),
    "n_shared_modules": int(len(shared_modules)),
    "n_train_modules": int(len(train_modules)),
    "n_shared_presentations": int(len(shared_presentations)),
    "n_train_presentations": int(len(train_presentations)),
    "n_shared_module_presentations": int(len(shared_modpres)),
    "n_train_module_presentations": int(len(train_modpres)),
    "contextual_interpretation": "shared curricular context across train and test",
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved:")
print(f"- {path_compact}")
print(f"- {path_compact_paper}")
print(f"- {path_commentary}")
print(f"- {path_metadata}")

### C2.3 — Context-Held-Out Split Design

### Funcao: module_coverage_penalty

Definicao isolada de module_coverage_penalty para reutilizacao nas etapas seguintes.


In [ ]:
# ============================================================
# C2.3 — Context-Held-Out Split Design
# REWRITTEN VERSION
# ============================================================
# Purpose:
#   Define a stronger grouped split based on curricular context.
#   This step does NOT yet materialize the final train/test tables.
#   It designs and audits a grouped holdout strategy using a
#   constrained search over candidate group subsets.
#
# Grouping unit:
#   (code_module, code_presentation)
#
# Main outputs:
#   - table_p13_3_group_inventory.csv
#   - table_p13_3_group_split_design.csv
#   - table_p13_3_split_design_summary.csv
#   - table_p13_3_candidate_solutions.csv
#   - metadata_p13_3_context_heldout_split_design.json
# ============================================================

import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.3 — Context-Held-Out Split Design")
print("=" * 70)
print("Rewritten version with constrained subset search.")

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

if "enrollment_survival_ready" not in available_tables:
    raise RuntimeError(
        "DuckDB table 'enrollment_survival_ready' not found. "
        "Run the corrected P5 first."
    )

df = con.execute("""
    SELECT *
    FROM enrollment_survival_ready
""").fetchdf()

required_cols = [
    "id_student", "code_module", "code_presentation",
    "event_observed", "t_final_week"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(
        f"'enrollment_survival_ready' is missing required columns: {missing_cols}"
    )

df = df.copy()
df["group_id"] = (
    df["code_module"].astype(str) + "||" + df["code_presentation"].astype(str)
)

# ------------------------------------------------------------
# 1) Inventory of presentation groups
# ------------------------------------------------------------
group_inventory = (
    df.groupby(["code_module", "code_presentation", "group_id"], dropna=False)
    .agg(
        n_enrollments=("id_student", "size"),
        n_unique_students=("id_student", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_t_final_week=("t_final_week", "max"),
        median_t_final_week=("t_final_week", "median"),
    )
    .reset_index()
    .sort_values(["code_module", "code_presentation"])
    .reset_index(drop=True)
)

group_inventory["event_rate"] = group_inventory["event_rate"].astype(float)
group_inventory["max_t_final_week"] = pd.to_numeric(group_inventory["max_t_final_week"], errors="coerce")
group_inventory["median_t_final_week"] = pd.to_numeric(group_inventory["median_t_final_week"], errors="coerce")

total_enrollments = int(len(df))
total_events = int(pd.to_numeric(df["event_observed"], errors="coerce").fillna(0).sum())
overall_event_rate = total_events / total_enrollments
all_modules = sorted(group_inventory["code_module"].astype(str).unique().tolist())
n_total_modules = len(all_modules)

print(f"Total enrollments: {total_enrollments}")
print(f"Overall event rate: {overall_event_rate:.6f}")
print(f"Total groups: {len(group_inventory)}")
print(f"Distinct modules: {n_total_modules}")

# ------------------------------------------------------------
# 2) Search configuration
# ------------------------------------------------------------
target_test_share = 0.30
target_test_n = int(round(total_enrollments * target_test_share))

# Candidate number of groups in the test split.
# With 22 groups total, this is still manageable combinatorially.
candidate_k_values = [4, 5, 6, 7]

# Score weights
w_share = 1.00
w_event = 1.00
w_module = 0.35
w_overrun = 0.15

# Optional penalty if the known duration-overrun context goes to test
special_overrun_group = "FFF||2013J"

# Module coverage target for the test set:
# reward broader curricular coverage.
def module_coverage_penalty(test_modules_count: int, total_modules_count: int) -> float:
    # penalty decreases as the number of distinct modules in test increases
    return 1.0 - (test_modules_count / total_modules_count)

# ------------------------------------------------------------
# 3) Evaluate candidate subsets
# ------------------------------------------------------------
records = []

groups_as_records = group_inventory.to_dict(orient="records")

for k in candidate_k_values:
    for combo in itertools.combinations(groups_as_records, k):
        test_group_ids = [x["group_id"] for x in combo]
        test_modules = sorted({str(x["code_module"]) for x in combo})

        test_n = int(sum(int(x["n_enrollments"]) for x in combo))
        test_events = int(sum(int(x["n_events"]) for x in combo))
        test_share = test_n / total_enrollments
        test_event_rate = test_events / test_n if test_n > 0 else np.nan

        share_error = abs(test_share - target_test_share)
        event_error = abs(test_event_rate - overall_event_rate) if pd.notna(test_event_rate) else np.inf
        module_pen = module_coverage_penalty(len(test_modules), n_total_modules)
        overrun_pen = 1.0 if special_overrun_group in test_group_ids else 0.0

        # hard guardrails to avoid clearly poor solutions
        if not (0.25 <= test_share <= 0.35):
            continue

        score = (
            w_share * share_error
            + w_event * event_error
            + w_module * module_pen
            + w_overrun * overrun_pen
        )

        records.append({
            "k_groups": k,
            "test_group_ids": " | ".join(sorted(test_group_ids)),
            "test_modules": " | ".join(test_modules),
            "n_test_groups": len(test_group_ids),
            "n_test_modules": len(test_modules),
            "test_n": test_n,
            "test_share": test_share,
            "test_events": test_events,
            "test_event_rate": test_event_rate,
            "share_error_abs": share_error,
            "event_rate_error_abs": event_error,
            "module_coverage_penalty": module_pen,
            "includes_special_overrun_group": int(special_overrun_group in test_group_ids),
            "score": score,
        })

candidate_solutions = pd.DataFrame(records)

if candidate_solutions.empty:
    raise RuntimeError(
        "No candidate grouped split satisfied the search constraints. "
        "Relax the candidate_k_values or the test_share guardrails."
    )

candidate_solutions = (
    candidate_solutions
    .sort_values(
        [
            "score",
            "share_error_abs",
            "event_rate_error_abs",
            "includes_special_overrun_group",
            "n_test_modules",
            "test_n",
        ],
        ascending=[True, True, True, True, False, True]
    )
    .reset_index(drop=True)
)

best_solution = candidate_solutions.iloc[0].to_dict()
best_test_group_ids = best_solution["test_group_ids"].split(" | ")

print("\nBest solution found:")
for k, v in best_solution.items():
    print(f"{k}: {v}")

# ------------------------------------------------------------
# 4) Materialize design table (design only, not split tables yet)
# ------------------------------------------------------------
design = group_inventory.copy()
design["is_test_group"] = design["group_id"].isin(best_test_group_ids).astype(int)
design["split_role"] = np.where(design["is_test_group"] == 1, "test", "train")
design = design.sort_values(["split_role", "code_module", "code_presentation"]).reset_index(drop=True)

# ------------------------------------------------------------
# 5) Summary of selected design
# ------------------------------------------------------------
summary_rows = []

for role in ["train", "test"]:
    tmp = design.loc[design["split_role"] == role].copy()
    summary_rows.append({
        "split_role": role,
        "n_groups": int(len(tmp)),
        "n_modules": int(tmp["code_module"].astype(str).nunique()),
        "n_enrollments": int(tmp["n_enrollments"].sum()),
        "n_unique_students_sum_over_groups": int(tmp["n_unique_students"].sum()),
        "n_events": int(tmp["n_events"].sum()),
        "event_rate_weighted": float(tmp["n_events"].sum() / tmp["n_enrollments"].sum()) if tmp["n_enrollments"].sum() > 0 else np.nan,
        "min_group_size": int(tmp["n_enrollments"].min()) if len(tmp) > 0 else np.nan,
        "median_group_size": float(tmp["n_enrollments"].median()) if len(tmp) > 0 else np.nan,
        "max_group_size": int(tmp["n_enrollments"].max()) if len(tmp) > 0 else np.nan,
        "max_followup_week": pd.to_numeric(tmp["max_t_final_week"], errors="coerce").max() if len(tmp) > 0 else np.nan,
    })

split_design_summary_role = pd.DataFrame(summary_rows)

overall_summary = pd.DataFrame([{
    "total_enrollments": int(total_enrollments),
    "overall_event_rate": float(overall_event_rate),
    "target_test_share": float(target_test_share),
    "target_test_n": int(target_test_n),
    "actual_test_n": int(design.loc[design["split_role"] == "test", "n_enrollments"].sum()),
    "actual_test_share": float(
        design.loc[design["split_role"] == "test", "n_enrollments"].sum() / total_enrollments
    ),
    "n_total_groups": int(len(design)),
    "n_test_groups": int((design["split_role"] == "test").sum()),
    "n_train_groups": int((design["split_role"] == "train").sum()),
    "n_total_modules": int(n_total_modules),
    "n_test_modules": int(design.loc[design["split_role"] == "test", "code_module"].astype(str).nunique()),
    "n_train_modules": int(design.loc[design["split_role"] == "train", "code_module"].astype(str).nunique()),
    "best_score": float(best_solution["score"]),
    "best_share_error_abs": float(best_solution["share_error_abs"]),
    "best_event_rate_error_abs": float(best_solution["event_rate_error_abs"]),
    "best_includes_special_overrun_group": int(best_solution["includes_special_overrun_group"]),
}])

table_p13_3_split_design_summary = pd.concat(
    [overall_summary.assign(section="overall"), split_design_summary_role.assign(section="by_role")],
    ignore_index=True,
    sort=False
)

# keep top candidate solutions for audit
table_p13_3_candidate_solutions = candidate_solutions.head(25).copy()

# ------------------------------------------------------------
# 6) Save outputs
# ------------------------------------------------------------
path_inventory = OUT_TABLES / "table_p13_3_group_inventory.csv"
path_design = OUT_TABLES / "table_p13_3_group_split_design.csv"
path_summary = OUT_TABLES / "table_p13_3_split_design_summary.csv"
path_candidates = OUT_TABLES / "table_p13_3_candidate_solutions.csv"
path_metadata = OUT_METADATA / "metadata_p13_3_context_heldout_split_design.json"

group_inventory.to_csv(path_inventory, index=False)
design.to_csv(path_design, index=False)
table_p13_3_split_design_summary.to_csv(path_summary, index=False)
table_p13_3_candidate_solutions.to_csv(path_candidates, index=False)

metadata = {
    "step": "P13.3",
    "title": "Context-Held-Out Split Design",
    "version": "rewritten_subset_search",
    "grouping_unit": ["code_module", "code_presentation"],
    "design_type": "subset_search_group_holdout",
    "target_test_share": target_test_share,
    "target_test_n": target_test_n,
    "overall_event_rate": float(overall_event_rate),
    "candidate_k_values": candidate_k_values,
    "score_weights": {
        "w_share": w_share,
        "w_event": w_event,
        "w_module": w_module,
        "w_overrun": w_overrun,
    },
    "special_overrun_group": special_overrun_group,
    "selected_test_group_ids": best_test_group_ids,
    "actual_test_n": int(design.loc[design["split_role"] == "test", "n_enrollments"].sum()),
    "actual_test_share": float(
        design.loc[design["split_role"] == "test", "n_enrollments"].sum() / total_enrollments
    ),
    "n_total_groups": int(len(design)),
    "n_test_groups": int((design["split_role"] == "test").sum()),
    "n_train_groups": int((design["split_role"] == "train").sum()),
    "notes": [
        "whole (code_module, code_presentation) groups are held out together",
        "test groups are never seen in training",
        "subset search optimizes approximate test share, event-rate proximity, and module diversity",
        "this step defines the split design but does not yet materialize split-specific benchmark tables"
    ],
    "output_tables": [
        path_inventory.as_posix(),
        path_design.as_posix(),
        path_summary.as_posix(),
        path_candidates.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 7) Display
# ------------------------------------------------------------
print("\nGroup inventory:")
display(group_inventory)

print("\nSelected grouped split design:")
display(design)

print("\nSplit design summary:")
display(table_p13_3_split_design_summary)

print("\nTop candidate solutions:")
display(table_p13_3_candidate_solutions)

print("\nSaved:")
print("-", path_inventory.resolve())
print("-", path_design.resolve())
print("-", path_summary.resolve())
print("-", path_candidates.resolve())
print("-", path_metadata.resolve())

### C2.4 — Materialize Context-Held-Out Partitions

In [ ]:
# ============================================================
# C2.4 — Materialize Context-Held-Out Partitions
# ============================================================
# Purpose:
#   Materialize the grouped context-held-out split approved in P13.3.
#
# Required inputs:
#   - DuckDB table: enrollment_survival_ready
#   - CSV from P13.3: table_p13_3_group_split_design.csv
#
# Main outputs:
#   - DuckDB table: enrollment_survival_ready_context_split
#   - table_p13_4_context_split_assignment_audit.csv
#   - table_p13_4_context_split_overlap_audit.csv
#   - table_p13_4_context_split_group_summary.csv
#   - table_p13_4_context_split_partition_summary.csv
#   - metadata_p13_4_context_split_materialization.json
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("C2.4 — Materialize Context-Held-Out Partitions")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

if "enrollment_survival_ready" not in available_tables:
    raise RuntimeError(
        "DuckDB table 'enrollment_survival_ready' not found. "
        "Run the corrected P5 first."
    )

path_design = OUT_TABLES / "table_p13_3_group_split_design.csv"
if not path_design.exists():
    raise FileNotFoundError(
        f"Required design file not found: {path_design.resolve()}. "
        "Run P13.3 first."
    )

# ------------------------------------------------------------
# 1) Load canonical enrollment table and approved design
# ------------------------------------------------------------
enr = con.execute("""
    SELECT *
    FROM enrollment_survival_ready
""").fetchdf()

design = pd.read_csv(path_design)

required_enr_cols = [
    "id_student", "code_module", "code_presentation",
    "event_observed", "t_final_week"
]
missing_enr_cols = [c for c in required_enr_cols if c not in enr.columns]
if missing_enr_cols:
    raise KeyError(
        f"'enrollment_survival_ready' is missing required columns: {missing_enr_cols}"
    )

required_design_cols = [
    "code_module", "code_presentation", "group_id", "is_test_group", "split_role"
]
missing_design_cols = [c for c in required_design_cols if c not in design.columns]
if missing_design_cols:
    raise KeyError(
        f"P13.3 design file is missing required columns: {missing_design_cols}"
    )

enr = enr.copy()
enr["group_id"] = (
    enr["code_module"].astype(str) + "||" + enr["code_presentation"].astype(str)
)

design = design.copy()
design["group_id"] = design["group_id"].astype(str)
design["split_role"] = design["split_role"].astype(str)
design["is_test_group"] = pd.to_numeric(design["is_test_group"], errors="coerce").fillna(0).astype(int)

# ------------------------------------------------------------
# 2) Merge split assignment onto enrollment table
# ------------------------------------------------------------
context_split = enr.merge(
    design[["group_id", "code_module", "code_presentation", "is_test_group", "split_role"]],
    on=["group_id", "code_module", "code_presentation"],
    how="left",
    validate="many_to_one"
)

missing_assignment_mask = context_split["split_role"].isna()
n_missing_assignments = int(missing_assignment_mask.sum())

if n_missing_assignments > 0:
    raise RuntimeError(
        f"{n_missing_assignments} enrollments did not receive a context split assignment. "
        "P13.3 design and enrollment_survival_ready are not aligned."
    )

context_split["context_split_name"] = "context_heldout_v1"
context_split["context_is_test"] = (context_split["split_role"] == "test").astype(int)
context_split["context_is_train"] = (context_split["split_role"] == "train").astype(int)

# ------------------------------------------------------------
# 3) Persist canonical DuckDB table
# ------------------------------------------------------------
con.register("tmp_enrollment_survival_ready_context_split_df", context_split)
con.execute("DROP TABLE IF EXISTS enrollment_survival_ready_context_split")
con.execute("""
    CREATE TABLE enrollment_survival_ready_context_split AS
    SELECT *
    FROM tmp_enrollment_survival_ready_context_split_df
""")

# ------------------------------------------------------------
# 4) Partition summary
# ------------------------------------------------------------
partition_summary = (
    context_split.groupby("split_role", dropna=False)
    .agg(
        n_enrollments=("id_student", "size"),
        n_unique_students=("id_student", "nunique"),
        n_groups=("group_id", "nunique"),
        n_modules=("code_module", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_t_final_week=("t_final_week", "max"),
        median_t_final_week=("t_final_week", "median"),
    )
    .reset_index()
    .sort_values("split_role")
    .reset_index(drop=True)
)

overall_partition_summary = pd.DataFrame([{
    "context_split_name": "context_heldout_v1",
    "n_total_enrollments": int(len(context_split)),
    "n_total_groups": int(context_split["group_id"].nunique()),
    "n_total_modules": int(context_split["code_module"].nunique()),
    "n_missing_assignments": n_missing_assignments,
    "test_share": float((context_split["split_role"] == "test").mean()),
    "overall_event_rate": float(pd.to_numeric(context_split["event_observed"], errors="coerce").fillna(0).mean()),
}])

table_p13_4_context_split_partition_summary = pd.concat(
    [overall_partition_summary.assign(section="overall"),
     partition_summary.assign(section="by_role")],
    ignore_index=True,
    sort=False
)

# ------------------------------------------------------------
# 5) Group-level summary
# ------------------------------------------------------------
table_p13_4_context_split_group_summary = (
    context_split.groupby(
        ["split_role", "code_module", "code_presentation", "group_id"],
        dropna=False
    )
    .agg(
        n_enrollments=("id_student", "size"),
        n_unique_students=("id_student", "nunique"),
        n_events=("event_observed", "sum"),
        event_rate=("event_observed", "mean"),
        max_t_final_week=("t_final_week", "max"),
        median_t_final_week=("t_final_week", "median"),
    )
    .reset_index()
    .sort_values(["split_role", "code_module", "code_presentation"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6) Overlap audit
# ------------------------------------------------------------
train_groups = set(
    context_split.loc[context_split["split_role"] == "train", "group_id"].astype(str).unique().tolist()
)
test_groups = set(
    context_split.loc[context_split["split_role"] == "test", "group_id"].astype(str).unique().tolist()
)
group_overlap = sorted(train_groups.intersection(test_groups))

train_modules = set(
    context_split.loc[context_split["split_role"] == "train", "code_module"].astype(str).unique().tolist()
)
test_modules = set(
    context_split.loc[context_split["split_role"] == "test", "code_module"].astype(str).unique().tolist()
)
module_overlap = sorted(train_modules.intersection(test_modules))

table_p13_4_context_split_overlap_audit = pd.DataFrame([{
    "context_split_name": "context_heldout_v1",
    "n_train_groups": len(train_groups),
    "n_test_groups": len(test_groups),
    "n_group_overlap_between_train_and_test": len(group_overlap),
    "group_overlap_values": " | ".join(group_overlap),
    "n_train_modules": len(train_modules),
    "n_test_modules": len(test_modules),
    "n_module_overlap_between_train_and_test": len(module_overlap),
    "module_overlap_values": " | ".join(module_overlap),
    "group_overlap_is_zero": len(group_overlap) == 0,
    "module_overlap_expected_and_allowed": True,
}])

# ------------------------------------------------------------
# 7) Assignment audit
# ------------------------------------------------------------
assignment_audit = pd.DataFrame([{
    "context_split_name": "context_heldout_v1",
    "n_rows_in_context_table": int(len(context_split)),
    "n_unique_groups_in_context_table": int(context_split["group_id"].nunique()),
    "n_unique_modules_in_context_table": int(context_split["code_module"].nunique()),
    "n_missing_assignments": n_missing_assignments,
    "all_rows_assigned": n_missing_assignments == 0,
    "n_test_rows": int((context_split["split_role"] == "test").sum()),
    "n_train_rows": int((context_split["split_role"] == "train").sum()),
    "test_share": float((context_split["split_role"] == "test").mean()),
    "event_rate_train": float(
        pd.to_numeric(
            context_split.loc[context_split["split_role"] == "train", "event_observed"],
            errors="coerce"
        ).fillna(0).mean()
    ),
    "event_rate_test": float(
        pd.to_numeric(
            context_split.loc[context_split["split_role"] == "test", "event_observed"],
            errors="coerce"
        ).fillna(0).mean()
    ),
}])

table_p13_4_context_split_assignment_audit = assignment_audit

# ------------------------------------------------------------
# 8) Save outputs
# ------------------------------------------------------------
path_assignment_audit = OUT_TABLES / "table_p13_4_context_split_assignment_audit.csv"
path_overlap_audit = OUT_TABLES / "table_p13_4_context_split_overlap_audit.csv"
path_group_summary = OUT_TABLES / "table_p13_4_context_split_group_summary.csv"
path_partition_summary = OUT_TABLES / "table_p13_4_context_split_partition_summary.csv"
path_metadata = OUT_METADATA / "metadata_p13_4_context_split_materialization.json"

table_p13_4_context_split_assignment_audit.to_csv(path_assignment_audit, index=False)
table_p13_4_context_split_overlap_audit.to_csv(path_overlap_audit, index=False)
table_p13_4_context_split_group_summary.to_csv(path_group_summary, index=False)
table_p13_4_context_split_partition_summary.to_csv(path_partition_summary, index=False)

metadata = {
    "step": "P13.4",
    "title": "Materialize Context-Held-Out Partitions",
    "context_split_name": "context_heldout_v1",
    "source_design_file": path_design.as_posix(),
    "source_enrollment_table": "enrollment_survival_ready",
    "output_duckdb_table": "enrollment_survival_ready_context_split",
    "n_total_rows": int(len(context_split)),
    "n_total_groups": int(context_split["group_id"].nunique()),
    "n_total_modules": int(context_split["code_module"].nunique()),
    "n_missing_assignments": n_missing_assignments,
    "group_overlap_is_zero": len(group_overlap) == 0,
    "selected_test_groups": sorted(test_groups),
    "notes": [
        "whole (code_module, code_presentation) groups are assigned together",
        "group overlap between train and test must be zero",
        "module overlap is expected and allowed because the design is presentation-held-out, not module-held-out",
        "this table does not replace the main benchmark split"
    ],
    "output_tables": [
        path_assignment_audit.as_posix(),
        path_overlap_audit.as_posix(),
        path_group_summary.as_posix(),
        path_partition_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 9) Display
# ------------------------------------------------------------
print("\nContext split assignment audit:")
display(table_p13_4_context_split_assignment_audit)

print("\nContext split overlap audit:")
display(table_p13_4_context_split_overlap_audit)

print("\nContext split partition summary:")
display(table_p13_4_context_split_partition_summary)

print("\nContext split group summary:")
display(table_p13_4_context_split_group_summary)

print("\nSaved:")
print("-", path_assignment_audit.resolve())
print("-", path_overlap_audit.resolve())
print("-", path_group_summary.resolve())
print("-", path_partition_summary.resolve())
print("-", path_metadata.resolve())

## C3 — Prepare the Preprocessing Layer for Modeling

In [ ]:
# ==============================================================
# C3 — Prepare the Preprocessing Layer for Modeling
# --------------------------------------------------------------
# Purpose:
#   Materialize train/test datasets for modeling from the
#   split-propagated modeling input tables.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - pp_linear_hazard_input
#   - pp_neural_hazard_input
#   - enrollment_cox_input_configured
#   - enrollment_deepsurv_input_configured
#
# Canonical naming policy in this step:
#   - pp_linear_hazard_input_split
#   - pp_neural_hazard_input_split
#   - enrollment_cox_input_configured_split
#   - enrollment_deepsurv_input_configured_split
#
# Outputs:
#   - DuckDB tables:
#       * pp_linear_hazard_ready_train
#       * pp_linear_hazard_ready_test
#       * pp_neural_hazard_ready_train
#       * pp_neural_hazard_ready_test
#       * enrollment_cox_ready_train
#       * enrollment_cox_ready_test
#       * enrollment_deepsurv_ready_train
#       * enrollment_deepsurv_ready_test
#   - audit summary tables
# ==============================================================

print("\n" + "=" * 70)
print("C3 — Prepare the Preprocessing Layer for Modeling")
print("=" * 70)

print("Methodological note: this step materializes train/test datasets only.")
print("No learned preprocessing transformation is fitted here.")
print("All datasets are derived from split-propagated input tables.")

import pandas as pd
from pathlib import Path

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["con", "OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

# ------------------------------
# 2) Canonicalize split table names (no fallback in materialization)
# ------------------------------
base_to_split = {
    "pp_linear_hazard_input": "pp_linear_hazard_input_split",
    "pp_neural_hazard_input": "pp_neural_hazard_input_split",
    "enrollment_cox_input_configured": "enrollment_cox_input_configured_split",
    "enrollment_deepsurv_input_configured": "enrollment_deepsurv_input_configured_split",
}

missing_base = [t for t in base_to_split if t not in available_tables]
if missing_base:
    raise RuntimeError(
        "Missing required upstream input table(s): " + ", ".join(missing_base)
    )

for base_table, split_table in base_to_split.items():
    con.execute(f"DROP TABLE IF EXISTS {split_table}")
    con.execute(f"""
    CREATE TABLE {split_table} AS
    SELECT *
    FROM {base_table}
    """)

# Strict requirement: only canonical *_split names from here onward
required_tables = [
    "pp_linear_hazard_input_split",
    "pp_neural_hazard_input_split",
    "enrollment_cox_input_configured_split",
    "enrollment_deepsurv_input_configured_split",
]

available_tables = set(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)
missing_tables = [t for t in required_tables if t not in available_tables]
if missing_tables:
    raise RuntimeError(
        "Missing required split-propagated input table(s): "
        + ", ".join(missing_tables)
    )

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 3) Materialize ready train/test tables
# ------------------------------
materialization_specs = [
    ("pp_linear_hazard_input_split", "pp_linear_hazard_ready_train", "train"),
    ("pp_linear_hazard_input_split", "pp_linear_hazard_ready_test", "test"),
    ("pp_neural_hazard_input_split", "pp_neural_hazard_ready_train", "train"),
    ("pp_neural_hazard_input_split", "pp_neural_hazard_ready_test", "test"),
    ("enrollment_cox_input_configured_split", "enrollment_cox_ready_train", "train"),
    ("enrollment_cox_input_configured_split", "enrollment_cox_ready_test", "test"),
    ("enrollment_deepsurv_input_configured_split", "enrollment_deepsurv_ready_train", "train"),
    ("enrollment_deepsurv_input_configured_split", "enrollment_deepsurv_ready_test", "test"),
]

for source_table, target_table, split_value in materialization_specs:
    con.execute(f"DROP TABLE IF EXISTS {target_table}")
    con.execute(f"""
    CREATE TABLE {target_table} AS
    SELECT *
    FROM {source_table}
    WHERE split = '{split_value}'
    """)

# ------------------------------
# 4) Audit summary
# ------------------------------
ready_tables = [
    "pp_linear_hazard_ready_train",
    "pp_linear_hazard_ready_test",
    "pp_neural_hazard_ready_train",
    "pp_neural_hazard_ready_test",
    "enrollment_cox_ready_train",
    "enrollment_cox_ready_test",
    "enrollment_deepsurv_ready_train",
    "enrollment_deepsurv_ready_test",
]

audit_rows = []

for table_name in ready_tables:
    cols = con.execute(f"PRAGMA table_info('{table_name}')").fetchdf()["name"].astype(str).tolist()
    n_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    n_distinct_enrollments = con.execute(
        f"SELECT COUNT(DISTINCT enrollment_id) FROM {table_name}"
    ).fetchone()[0]

    split_values = con.execute(f"""
    SELECT STRING_AGG(DISTINCT split, ', ' ORDER BY split)
    FROM {table_name}
    """).fetchone()[0]

    target_col = None
    target_sum = None
    duration_min = None
    duration_max = None

    if "event_t" in cols:
        target_col = "event_t"
        target_sum = con.execute(f"SELECT COALESCE(SUM(event_t), 0) FROM {table_name}").fetchone()[0]

    elif "event" in cols:
        target_col = "event"
        target_sum = con.execute(f"SELECT COALESCE(SUM(event), 0) FROM {table_name}").fetchone()[0]

    if "duration" in cols:
        duration_min, duration_max = con.execute(
            f"SELECT MIN(duration), MAX(duration) FROM {table_name}"
        ).fetchone()

    audit_rows.append({
        "table_name": table_name,
        "n_rows": int(n_rows),
        "n_distinct_enrollments": int(n_distinct_enrollments),
        "n_columns": len(cols),
        "split_values_present": split_values,
        "target_column": target_col,
        "target_sum": float(target_sum) if target_sum is not None else None,
        "duration_min": float(duration_min) if duration_min is not None else None,
        "duration_max": float(duration_max) if duration_max is not None else None,
    })

audit_df = pd.DataFrame(audit_rows)

audit_path = TABLES_DIR / "table_p14_ready_dataset_audit.csv"
audit_df.to_csv(audit_path, index=False)

# ------------------------------
# 5) Export ready tables
# ------------------------------
for table_name in ready_tables:
    parquet_path = DATA_OUTPUT_DIR / f"{table_name}.parquet"
    csv_path = DATA_OUTPUT_DIR / f"{table_name}.csv"
    con.execute(f"COPY {table_name} TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
    con.execute(f"COPY {table_name} TO '{csv_path.as_posix()}' (HEADER, DELIMITER ',')")

# ------------------------------
# 6) Output
# ------------------------------
print("\nCanonical split tables materialized:")
for base_table, split_table in base_to_split.items():
    print(f"- {base_table} -> {split_table}")

print("\nReady dataset audit summary:")
display(audit_df)

print("\nSaved:")
print("-", audit_path.resolve())

print("\nExported tables:")
for table_name in ready_tables:
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.parquet").resolve())
    print("-", (DATA_OUTPUT_DIR / f"{table_name}.csv").resolve())